# 🐟 Clasificación de Peces: CNN Propia vs ResNet50

**Dataset:** [A Large Scale Fish Dataset](https://www.kaggle.com/crowww/a-large-scale-fish-dataset)  
**Tarea:** Clasificación multiclase — 9 especies de peces

## Arquitecturas comparadas

| | Modelo A | Modelo B |
|---|---|---|
| **Backbone** | CNN propia (desde cero) | ResNet50 (preentrenado ImageNet) |
| **Clasificador** | MLP propia | La misma MLP |
| **Entrenamiento** | Todo desde cero | Transfer Learning |

La **MLP es idéntica** en ambos modelos. Así la única variable que cambia es el extractor de features (backbone), lo que hace la comparación justa.

## Justificación de decisiones

**¿Por qué Transfer Learning en ResNet50?**  
ResNet50 preentrenado en ImageNet ya aprendió features visuales generales (bordes, texturas, formas) útiles para cualquier tarea de visión. Con Transfer Learning:
- Fase 1: se congela el backbone, solo se entrena la MLP
- Fase 2: se hace fine-tuning del último bloque residual

**¿Por qué comparar contra CNN propia?**  
Para medir cuánto aporta el preentrenamiento: si la CNN propia se acerca a ResNet50, significa que el dataset es suficientemente grande y específico. Si ResNet50 gana ampliamente, confirma el valor del Transfer Learning.

**¿Por qué ResNet50 y no VGG o InceptionV3?**  
- VGG tiene ~138M parámetros vs ~25M de ResNet50, sin mejora notable
- InceptionV3 requiere mínimo 299×299, más costoso en Colab
- ResNet50 es el baseline estándar en papers de clasificación de imágenes

## 0. Instalación y Setup

In [ ]:
!pip install -q kaggle matplotlib seaborn scikit-learn

In [ ]:
from google.colab import files
import os

print('Subí tu kaggle.json (descargalo de kaggle.com → Settings → API → Create Token):')
uploaded = files.upload()
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json
print('✅ Kaggle API configurada')

In [ ]:
!kaggle datasets download -d crowww/a-large-scale-fish-dataset -p /content/fish_data --unzip
print('✅ Dataset descargado')

## 1. Imports y configuración global

In [ ]:
import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from torchvision.models import ResNet50_Weights
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# ── Hiperparámetros ──
IMG_SIZE     = 224
BATCH_SIZE   = 32
NUM_CLASSES  = 9

# Modelo A: CNN propia + MLP
EPOCHS_A     = 30
LR_A         = 1e-3

# Modelo B: ResNet50 + MLP
EPOCHS_B1    = 10   # Fase 1: solo MLP
EPOCHS_B2    = 10   # Fase 2: fine-tuning
LR_B1        = 1e-3
LR_B2        = 1e-4

## 2. Exploración del dataset

In [ ]:
DATA_ROOT = Path('/content/fish_data')

def find_class_root(base):
    for p in sorted(base.rglob('*')):
        if p.is_dir():
            children = [c for c in p.iterdir() if c.is_dir()]
            if len(children) >= 5:
                imgs = list(children[0].glob('*.png')) + list(children[0].glob('*.jpg'))
                if imgs:
                    return p
    return base

CLASS_ROOT = find_class_root(DATA_ROOT)
print(f'Directorio de clases: {CLASS_ROOT}\n')

classes = sorted([d.name for d in CLASS_ROOT.iterdir() if d.is_dir()])
print(f'Clases ({len(classes)}):')
for i, c in enumerate(classes):
    n = len(list((CLASS_ROOT/c).glob('*.png'))) + len(list((CLASS_ROOT/c).glob('*.jpg')))
    print(f'  [{i}] {c}: {n} imágenes')

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
fig.suptitle('Una muestra por clase — Fish Dataset', fontsize=15, fontweight='bold')
for ax, cls in zip(axes.flat, classes):
    imgs = list((CLASS_ROOT/cls).glob('*.png')) + list((CLASS_ROOT/cls).glob('*.jpg'))
    if imgs:
        ax.imshow(Image.open(imgs[0]))
        ax.set_title(cls, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.savefig('samples.png', dpi=100)
plt.show()

## 3. Preprocesamiento y DataLoaders

In [ ]:
# Estadísticas ImageNet — se usan en ambos modelos para comparación justa
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class TransformSubset(torch.utils.data.Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform
    def __len__(self): return len(self.subset)
    def __getitem__(self, i):
        x, y = self.subset[i]
        return self.transform(x), y

full_ds = datasets.ImageFolder(str(CLASS_ROOT))
CLASS_NAMES = full_ds.classes
N = len(full_ds)

g = torch.Generator().manual_seed(SEED)
n_train = int(0.70 * N)
n_val   = int(0.15 * N)
n_test  = N - n_train - n_val
idx = torch.randperm(N, generator=g).tolist()
train_idx = idx[:n_train]
val_idx   = idx[n_train:n_train+n_val]
test_idx  = idx[n_train+n_val:]

raw_ds = datasets.ImageFolder(str(CLASS_ROOT), transform=transforms.Lambda(lambda x: x))

train_ds = TransformSubset(Subset(raw_ds, train_idx), train_tf)
val_ds   = TransformSubset(Subset(raw_ds, val_idx),   val_tf)
test_ds  = TransformSubset(Subset(raw_ds, test_idx),  val_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Total: {N} | Train: {n_train} | Val: {n_val} | Test: {n_test}')
print(f'Clases: {CLASS_NAMES}')

## 4. Definición de la MLP (compartida por ambos modelos)

In [ ]:
class FishMLP(nn.Module):
    """
    Cabeza clasificadora MLP compartida por ambos modelos.
    Recibe un vector de features del backbone y produce logits para 9 clases.

    Arquitectura:
        in_features → 512 → 256 → 128 → num_classes
    Con BatchNorm, ReLU y Dropout en cada capa oculta.
    """
    def __init__(self, in_features, num_classes=9):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),

            nn.Linear(128, num_classes)
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x)

print('MLP definida:')
print('  in_features → 512 → 256 → 128 → 9 clases')
print('  (BatchNorm + ReLU + Dropout en cada capa oculta)')

## 5. Modelo A: CNN Propia + MLP

La CNN es un extractor de features hecho desde cero, con 5 bloques convolucionales.
Cada bloque tiene: Conv2d → BatchNorm → ReLU → Conv2d → BatchNorm → ReLU → MaxPool.
Al final, un Global Average Pooling convierte el mapa de features en un vector 1D
que entra a la MLP para clasificar.

In [ ]:
class ConvBlock(nn.Module):
    """Bloque convolucional doble: Conv-BN-ReLU-Conv-BN-ReLU."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)


class FishCNN(nn.Module):
    """
    CNN propia para extracción de features.

    Input:  (B, 3, 224, 224)
    Block1: 3→32,   MaxPool → (B, 32,  112, 112)
    Block2: 32→64,  MaxPool → (B, 64,  56,  56 )
    Block3: 64→128, MaxPool → (B, 128, 28,  28 )
    Block4: 128→256,MaxPool → (B, 256, 14,  14 )
    Block5: 256→512,MaxPool → (B, 512, 7,   7  )
    GAP             →         (B, 512)
    """
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3,   32),  nn.MaxPool2d(2), nn.Dropout2d(0.1),
            ConvBlock(32,  64),  nn.MaxPool2d(2), nn.Dropout2d(0.1),
            ConvBlock(64,  128), nn.MaxPool2d(2), nn.Dropout2d(0.15),
            ConvBlock(128, 256), nn.MaxPool2d(2), nn.Dropout2d(0.15),
            ConvBlock(256, 512), nn.MaxPool2d(2),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)  # Global Average Pooling
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)          # (B, 512, 1, 1)
        return x.flatten(1)      # (B, 512)


class ModeloA(nn.Module):
    """CNN propia + MLP propia. Todo entrenado desde cero."""
    def __init__(self, num_classes=9):
        super().__init__()
        self.backbone = FishCNN()
        self.mlp      = FishMLP(in_features=512, num_classes=num_classes)

    def forward(self, x):
        features = self.backbone(x)   # (B, 512)
        return self.mlp(features)     # (B, 9)


modelo_a = ModeloA(NUM_CLASSES).to(DEVICE)
params_a  = sum(p.numel() for p in modelo_a.parameters())
print(f'Modelo A — parámetros totales: {params_a:,}')
print(f'  CNN backbone : {sum(p.numel() for p in modelo_a.backbone.parameters()):,}')
print(f'  MLP          : {sum(p.numel() for p in modelo_a.mlp.parameters()):,}')

## 6. Modelo B: ResNet50 + la misma MLP

In [ ]:
class ModeloB(nn.Module):
    """
    ResNet50 preentrenado en ImageNet como backbone + la misma MLP.
    Se remueve la capa FC original de ResNet50 y se reemplaza por la FishMLP.
    """
    def __init__(self, num_classes=9, freeze_backbone=True):
        super().__init__()
        # Cargar ResNet50 preentrenado, sin la FC original
        resnet = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        resnet.fc = nn.Identity()  # Quitar clasificador original → output (B, 2048)
        self.backbone = resnet

        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

        # ResNet50 sin FC produce features de 2048 dimensiones
        self.mlp = FishMLP(in_features=2048, num_classes=num_classes)

    def unfreeze_last_block(self):
        """Descongela layer4 de ResNet para fine-tuning (Fase 2)."""
        for p in self.backbone.layer4.parameters():
            p.requires_grad = True

    def forward(self, x):
        features = self.backbone(x)   # (B, 2048)
        return self.mlp(features)     # (B, 9)


modelo_b = ModeloB(NUM_CLASSES, freeze_backbone=True).to(DEVICE)
params_b_total = sum(p.numel() for p in modelo_b.parameters())
params_b_train = sum(p.numel() for p in modelo_b.parameters() if p.requires_grad)
print(f'Modelo B — parámetros totales    : {params_b_total:,}')
print(f'Modelo B — parámetros entrenables: {params_b_train:,} (Fase 1, backbone congelado)')
print(f'  ResNet50 backbone: {sum(p.numel() for p in modelo_b.backbone.parameters()):,}')
print(f'  MLP              : {sum(p.numel() for p in modelo_b.mlp.parameters()):,}')

## 7. Funciones de entrenamiento y evaluación

In [ ]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    loss_sum, correct, total = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        out  = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * X.size(0)
        correct  += (out.argmax(1) == y).sum().item()
        total    += X.size(0)
    return loss_sum / total, correct / total

@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        out  = model(X)
        loss = criterion(out, y)
        loss_sum += loss.item() * X.size(0)
        correct  += (out.argmax(1) == y).sum().item()
        total    += X.size(0)
    return loss_sum / total, correct / total

def training_loop(model, train_ldr, val_ldr, criterion, optimizer,
                  scheduler, epochs, name, save_path):
    history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}
    best_acc = 0.0
    t0 = time.time()
    for ep in range(1, epochs+1):
        tl, ta = train_epoch(model, train_ldr, criterion, optimizer)
        vl, va = eval_epoch(model, val_ldr,   criterion)
        if scheduler: scheduler.step(vl)
        history['train_loss'].append(tl); history['val_loss'].append(vl)
        history['train_acc'].append(ta);  history['val_acc'].append(va)
        if va > best_acc:
            best_acc = va
            torch.save(model.state_dict(), save_path)
        print(f'[{name}] Ep {ep:02d}/{epochs} | '
              f'Train {ta:.4f} ({tl:.4f}) | Val {va:.4f} ({vl:.4f})'
              f"{' ← best' if va == best_acc else ''}")
    elapsed = time.time() - t0
    print(f'⏱ {elapsed/60:.1f} min | Mejor val acc: {best_acc:.4f}')
    return history, elapsed

@torch.no_grad()
def evaluate_test(model, loader, class_names, title):
    model.eval()
    preds_all, labels_all = [], []
    for X, y in loader:
        preds_all.extend(model(X.to(DEVICE)).argmax(1).cpu().numpy())
        labels_all.extend(y.numpy())
    acc = accuracy_score(labels_all, preds_all)
    f1  = f1_score(labels_all, preds_all, average='weighted')
    print(f'\n=== {title} ===')
    print(f'Test Accuracy : {acc:.4f}')
    print(f'Test F1 (weighted): {f1:.4f}')
    print(classification_report(labels_all, preds_all, target_names=class_names))
    return labels_all, preds_all, acc, f1

def plot_history(history, title):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ep = range(1, len(history['train_loss'])+1)
    ax1.plot(ep, history['train_loss'], 'b-o', ms=4, label='Train')
    ax1.plot(ep, history['val_loss'],   'r-o', ms=4, label='Val')
    ax1.set_title('Loss'); ax1.legend(); ax1.grid(alpha=0.3)
    ax2.plot(ep, history['train_acc'], 'b-o', ms=4, label='Train')
    ax2.plot(ep, history['val_acc'],   'r-o', ms=4, label='Val')
    ax2.set_title('Accuracy'); ax2.set_ylim(0,1); ax2.legend(); ax2.grid(alpha=0.3)
    fig.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{title.replace(" ","_")}.png', dpi=100)
    plt.show()

def plot_cm(labels, preds, class_names, title):
    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Matriz de Confusión — {title}', fontweight='bold')
    plt.ylabel('Real'); plt.xlabel('Predicho')
    plt.xticks(rotation=45, ha='right'); plt.tight_layout()
    plt.savefig(f'CM_{title.replace(" ","_")}.png', dpi=100)
    plt.show()

print('✅ Funciones definidas')

## 8. Entrenamiento Modelo A (CNN propia + MLP)

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

opt_a  = optim.AdamW(modelo_a.parameters(), lr=LR_A, weight_decay=1e-4)
sch_a  = optim.lr_scheduler.ReduceLROnPlateau(opt_a, patience=4, factor=0.5)

print('='*60)
print('MODELO A: CNN Propia + MLP — entrenando desde cero')
print('='*60)
history_a, time_a = training_loop(
    modelo_a, train_loader, val_loader, criterion, opt_a, sch_a,
    EPOCHS_A, 'Modelo A', 'modelo_a_best.pth'
)

In [ ]:
plot_history(history_a, 'Modelo A — CNN Propia + MLP')
modelo_a.load_state_dict(torch.load('modelo_a_best.pth'))
labels_a, preds_a, acc_a, f1_a = evaluate_test(modelo_a, test_loader, CLASS_NAMES, 'Modelo A')
plot_cm(labels_a, preds_a, CLASS_NAMES, 'Modelo A')

## 9. Entrenamiento Modelo B (ResNet50 + MLP)

**Fase 1:** Backbone congelado, solo se entrena la MLP. Esto permite que la MLP
aprenda a interpretar los features de ResNet50 sin desestabilizar el backbone.

**Fase 2:** Se descongela el último bloque residual (`layer4`) para fine-tuning.
Se usa un learning rate más bajo para no destruir los pesos preentrenados.

In [ ]:
# ── FASE 1: Solo entrena la MLP ──
params_train_b1 = sum(p.numel() for p in modelo_b.parameters() if p.requires_grad)
print(f'Fase 1 — parámetros entrenables: {params_train_b1:,} (solo MLP)')

opt_b1 = optim.Adam(filter(lambda p: p.requires_grad, modelo_b.parameters()), lr=LR_B1)
sch_b1 = optim.lr_scheduler.ReduceLROnPlateau(opt_b1, patience=3, factor=0.5)

print('='*60)
print('MODELO B — FASE 1: ResNet50 congelado, entrenando MLP')
print('='*60)
history_b1, _ = training_loop(
    modelo_b, train_loader, val_loader, criterion, opt_b1, sch_b1,
    EPOCHS_B1, 'Modelo B Fase1', 'modelo_b_phase1.pth'
)

In [ ]:
# ── FASE 2: Fine-tuning del último bloque ──
modelo_b.unfreeze_last_block()
params_train_b2 = sum(p.numel() for p in modelo_b.parameters() if p.requires_grad)
print(f'Fase 2 — parámetros entrenables: {params_train_b2:,} (MLP + layer4)')

opt_b2 = optim.Adam(filter(lambda p: p.requires_grad, modelo_b.parameters()),
                    lr=LR_B2, weight_decay=1e-4)
sch_b2 = optim.lr_scheduler.CosineAnnealingLR(opt_b2, T_max=EPOCHS_B2)

print('='*60)
print('MODELO B — FASE 2: Fine-tuning layer4 + MLP')
print('='*60)
history_b2, time_b = training_loop(
    modelo_b, train_loader, val_loader, criterion, opt_b2, sch_b2,
    EPOCHS_B2, 'Modelo B Fase2', 'modelo_b_best.pth'
)

history_b = {k: history_b1[k] + history_b2[k] for k in history_b1}

In [ ]:
plot_history(history_b, 'Modelo B — ResNet50 + MLP')
modelo_b.load_state_dict(torch.load('modelo_b_best.pth'))
labels_b, preds_b, acc_b, f1_b = evaluate_test(modelo_b, test_loader, CLASS_NAMES, 'Modelo B')
plot_cm(labels_b, preds_b, CLASS_NAMES, 'Modelo B')

## 10. Comparación Final

In [ ]:
results = pd.DataFrame({
    'Modelo'              : ['Modelo A — CNN Propia + MLP', 'Modelo B — ResNet50 + MLP'],
    'Backbone'            : ['CNN desde cero', 'ResNet50 (ImageNet)'],
    'MLP'                 : ['FishMLP (512→256→128→9)', 'FishMLP (512→256→128→9)'],
    'Parámetros totales'  : [f'{params_a:,}', f'{params_b_total:,}'],
    'Epochs'              : [EPOCHS_A, EPOCHS_B1 + EPOCHS_B2],
    'Test Accuracy'       : [f'{acc_a:.4f}', f'{acc_b:.4f}'],
    'Test F1 (weighted)'  : [f'{f1_a:.4f}', f'{f1_b:.4f}'],
    'Tiempo entrenamiento': [f'{time_a/60:.1f} min', f'{time_b/60:.1f} min'],
})
print('\n' + '='*70)
print('TABLA COMPARATIVA FINAL')
print('='*70)
print(results.T.to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Comparación Final: Modelo A vs Modelo B', fontsize=15, fontweight='bold')
nombres = ['Modelo A\n(CNN propia + MLP)', 'Modelo B\n(ResNet50 + MLP)']
colores = ['#E53935', '#1E88E5']

for ax, vals, title, ylim in zip(
    axes,
    [[acc_a, acc_b], [f1_a, f1_b], [params_a/1e6, params_b_total/1e6]],
    ['Test Accuracy', 'F1 Score (Weighted)', 'Parámetros (M)'],
    [1, 1, None]
):
    bars = ax.bar(nombres, vals, color=colores, width=0.5, edgecolor='white', linewidth=1.5)
    ax.set_title(title, fontweight='bold')
    if ylim: ax.set_ylim(0, ylim)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + (0.01 if ylim else 0.3),
                f'{v:.4f}' if ylim else f'{v:.1f}M', ha='center', fontweight='bold', fontsize=10)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('comparacion_final.png', dpi=150)
plt.show()

In [ ]:
# ── Curvas de validación superpuestas ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ep_a = range(1, len(history_a['val_acc'])+1)
ep_b = range(1, len(history_b['val_acc'])+1)

ax1.plot(ep_a, history_a['val_loss'], 'r-', lw=2, label='Modelo A (CNN propia)')
ax1.plot(ep_b, history_b['val_loss'], 'b-', lw=2, label='Modelo B (ResNet50)')
ax1.set_title('Val Loss'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(ep_a, history_a['val_acc'], 'r-', lw=2, label='Modelo A (CNN propia)')
ax2.plot(ep_b, history_b['val_acc'], 'b-', lw=2, label='Modelo B (ResNet50)')
ax2.set_title('Val Accuracy'); ax2.set_ylim(0,1); ax2.legend(); ax2.grid(alpha=0.3)

fig.suptitle('Curvas de Validación Comparativas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('curvas_comparativas.png', dpi=100)
plt.show()

print('\n✅ Análisis completo. Todos los gráficos guardados.')

## 11. Conclusiones

### ¿Qué mide esta comparación?
Como la MLP es **idéntica** en ambos modelos, la única diferencia es el **backbone**:
- **Modelo A** usa una CNN entrenada desde cero → aprende features específicamente del dataset de peces
- **Modelo B** usa ResNet50 preentrenado → parte de features generales aprendidos en 14M imágenes de ImageNet

### Interpretación de resultados
| Resultado | Interpretación |
|-----------|----------------|
| Modelo B >> Modelo A | El preentrenamiento agrega mucho valor. El dataset de peces es insuficiente para aprender buenos features desde cero. |
| Modelo B ≈ Modelo A | El dataset de peces es suficientemente grande y específico. La CNN propia puede aprender features igual de buenos. |
| Modelo A converge más lento | Normal: aprender features desde cero requiere más epochs. |
| Modelo A tiene menos parámetros | La CNN propia es más liviana, lo que puede ser ventaja en deployment. |

### ¿Cuándo usar Transfer Learning?
- Dataset pequeño o dominio visualmente similar a ImageNet → **usar ResNet50 + TL**
- Dataset muy grande o dominio muy diferente (ej: imágenes médicas, satélite) → **CNN propia puede competir**
